# Test-fragment inference viewer for YoussefMoNader's v8-in model

A copy of `test_inference.ipynb` that runs the external v8-in model ([ink-8um-v8in](https://huggingface.co/YoussefMoNader/ink-8um-v8in)) exactly as released. The model is the author's `ink8um.InkDetector`: 24 × 64 × 64 uint8 tiles clipped to [0, 200] and scaled by 1/255, InstanceNorm3d, trilinear resize to 96 × 256 × 256, ResNet3D-50, max over depth, a 2D decoder, and 64 × 64 logits. It loads the released fp32 `model.safetensors` with `strict=True`. Stitching is the author's unmodified `ink8um.inference.predict_stack`: coverage mask, tiles fully inside it at stride 21, fp16 autocast, sigmoid, and Gaussian-weighted sum divided by tile count. No code from our campaign models (surface labels, norm stats, multitile head) is involved.

Each fragment is predicted once on the whole render frame, as `predict.py` would, using the author's default window: the centred 24 layers (2→26 of our 28). The result is then cropped to the same bounds as `test_inference.ipynb`. Composites, layout, and file names are unchanged. Weights live in `models/v8in/<variant>/` (`base` or `pherc1447_loo_w062`), and figures go to `output/test_visualizations/v8in_<variant>/`. Only the fast figure `<name>_fast.jpg` is rendered; it is previewed inline at 1/3 scale:

| centred 24 layers | composite (1 cm scale bar) |
|---|---|

- pixels no tile reached (0 in `predict_stack`) are drawn pale blue, like masked cells in the original notebook
- our zarrs already follow the training depth orientation (`depth_reversed.json`), so `REVERSE_LAYERS` stays `False` for both variants. The finetune's `config.json` sets `reverse_layers: true`, but that applies to the author's PHerc1447 renders, which our 1447 zarrs already correct.
- set `DRY_RUN = True` in cell 3 to inspect composites without loading the model


In [ ]:
import os
from pathlib import Path
# "base" (v8-in, zero-shot everywhere) or "pherc1447_loo_w062" (finetuned on PHerc1447 w058+w060)
V8_VARIANT = os.getenv("VESUVIUS_V8_VARIANT", "base")
V8_DIR = Path(os.getenv("VESUVIUS_V8_DIR", f"/vesuvius/models/v8in/{V8_VARIANT}"))
RUN_ID = f"v8in_{V8_VARIANT}"


In [2]:
# v8-in checkpoint and inference settings
import json
import os
from pathlib import Path

DRY_RUN = False
MODEL_PATH = V8_DIR / "model.safetensors"
V8_CONFIG_PATH = V8_DIR / "config.json"
if not MODEL_PATH.is_file() or not V8_CONFIG_PATH.is_file():
    raise FileNotFoundError(f"v8-in weights/config missing under {V8_DIR}")
with open(V8_CONFIG_PATH, "r", encoding="utf-8") as handle:
    V8_CONFIG = json.load(handle)
V8_DEPTH = int(V8_CONFIG["in_depth"])
V8_TILE = int(V8_CONFIG["tile_size"])
V8_CLIP = int(V8_CONFIG["clip_max"])
# our zarrs are already in training orientation; the finetune's reverse_layers=true is for the author's renders
REVERSE_LAYERS = False
# the author's stride and batch (config.json / predict.py defaults)
EVAL_STRIDE = int(V8_CONFIG["stride"])
BATCH_SIZE = 16
NUM_WORKERS = 4
# same crop alignment as test_inference.ipynb so crop bounds (and composites) match exactly
TILE_SIZE = 16
COMPOSITE_METHOD = "maxproj"
COMPOSITE_D0 = 10
COMPOSITE_D1 = 18
COMPOSITE_DISPLAY = "raw"
COMPOSITE_CLAHE = False
GENERATE_COMPOSITES = False
VOXEL_UM = 9.362
DISPLAY_SCALE = 0.25
MASK_CROP_MARGIN = 32
PAD_PX = 24
OUTPUT_DIR = "output"
COMPOSITE_CACHE = "output/composite_cache"
FRAGMENTS = {
    "auto_grown_20260814140748": 20260814140748,
    "auto_grown_20260717193517": 20260717193517,
    "auto_grown_20260720090842": 20260720090842,
    "auto_grown_20250703034159": 20250703034159,
    "auto_grown_20260723112922652_merged": 20260723112922,
    "auto_grown_20260921094413486": 20260921094413,
    "auto_grown_20260918132724424": 20260918132724,
    "auto_grown_20260922073234974": 20260922073234,
    "auto_grown_20260922161631422_merged": 20260922161631,
    "auto_grown_20260925085345806_abf": 20260925085345,
    "PHerc0800_z9264_w040_abf": 20260928000001,
    "PHerc0125_z15344_w040_abf": 20260928000002,
    "PHerc0211_z7312_w080_abf": 20260928000003,
}
# extra segment sets (assemble_test_segments.py --include-extras) are keyed by segment folder name;
# their masks, surface labels, and composites live in an "extras" subfolder and are synced through R2, not git
EXTRAS_SUBDIR = "extras"
EXTRAS_MESH_ROOT = os.path.join("tifxyz", "atlas")
QUICK_D_MESH_ROOT = os.path.join("tifxyz", "quick_patches_from_D")


def is_extra(scroll_id):
    return not str(scroll_id).isdigit()


def mask_dir_for(scroll_id):
    return os.path.join("masks", EXTRAS_SUBDIR) if is_extra(scroll_id) else "masks"


def mask_path_for(scroll_id):
    return os.path.join(mask_dir_for(scroll_id), f"{scroll_id}.png")


print(RUN_ID, MODEL_PATH, f"depth={V8_DEPTH}", f"tile={V8_TILE}", f"clip={V8_CLIP}", f"reverse={REVERSE_LAYERS}",
      f"test_scrolls={len(FRAGMENTS)}", f"stride={EVAL_STRIDE}")


v8in_base /vesuvius/models/v8in/base/model.safetensors depth=24 tile=64 clip=200 reverse=False test_scrolls=13 stride=21


In [3]:
import gc
import importlib
import os
import subprocess
import sys

REPO = os.getenv("VESUVIUS_REPO", "/vesuvius" if os.name == "posix" else r"C:\Users\ChenJeff\Documents\vesuvius")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

# ink8um.modeling needs safetensors + huggingface_hub (PyTorchModelHubMixin)
try:
    import huggingface_hub  # noqa: F401
    import safetensors  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "safetensors", "huggingface_hub>=0.25"], check=True)
from safetensors.torch import load_file as load_safetensors

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

if str(V8_DIR) not in sys.path:
    sys.path.insert(0, str(V8_DIR))
from ink8um import InkDetector
from ink8um.inference import coverage_mask, predict_stack, tile_positions

from utils.config import Config
from utils import visualizer as visualizer_module

visualizer_module = importlib.reload(visualizer_module)
load_or_create_midslice_mask = visualizer_module.load_or_create_midslice_mask

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(COMPOSITE_CACHE, exist_ok=True)
print(
    f"repo={REPO}",
    f"| torch={torch.__version__}",
    f"| cuda={torch.cuda.is_available()}",
    f"| dry_run={DRY_RUN}",
    f"| ink8um={sys.modules['ink8um'].__file__}",
)



[notice] A new release of pip is available: 23.3.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
2026-09-29 23:30:52.559040: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-29 23:30:52.602419: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-29 23:30:52.626084: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790724652.644279 1771193 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790724652.648820 1771193 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:179072

repo=/vesuvius | torch=2.1.0+cu118 | cuda=True | dry_run=False | ink8um=/vesuvius/models/v8in/base/ink8um/__init__.py


/vesuvius/utils/visualizer.py:60: UserWarning: Overwriting the cmap 'inferno_nan' that was already in the registry.
  _mpl.colormaps.register(_inferno_nan, name='inferno_nan', force=True)
/vesuvius/utils/visualizer.py:61: UserWarning: Overwriting the cmap 'ylgnbu_nan' that was already in the registry.
  _mpl.colormaps.register(_ylgnbu_nan, name='ylgnbu_nan', force=True)
/vesuvius/utils/visualizer.py:62: UserWarning: Overwriting the cmap 'purples_nan' that was already in the registry.
  _mpl.colormaps.register(_purples_nan, name='purples_nan', force=True)
/vesuvius/utils/visualizer.py:63: UserWarning: Overwriting the cmap 'ink_gray_nan' that was already in the registry.
  _mpl.colormaps.register(_ink_gray_nan, name='ink_gray_nan', force=True)


In [4]:
# composite helper and projection settings
def _project_depth(vol, d0, d1, method, bounds=None):
    """project only the requested spatial bounds over slices [d0, d1)"""
    _, height, width = map(int, vol.shape)
    y0, y1, x0, x1 = bounds or (0, height, 0, width)
    d0, d1 = max(0, d0), min(d1, int(vol.shape[0]))
    shape = (y1 - y0, x1 - x0)
    if method == "maxproj":
        acc = np.zeros(shape, np.float32)
        for depth in range(d0, d1):
            acc = np.maximum(acc, np.asarray(vol[depth, y0:y1, x0:x1]).astype(np.float32))
        return acc
    if method == "meanproj":
        acc = np.zeros(shape, np.float64)
        for depth in range(d0, d1):
            acc += np.asarray(vol[depth, y0:y1, x0:x1])
        return (acc / max(d1 - d0, 1)).astype(np.float32)
    if method == "minproj":
        acc = np.full(shape, np.inf, np.float32)
        for depth in range(d0, d1):
            acc = np.minimum(acc, np.asarray(vol[depth, y0:y1, x0:x1]).astype(np.float32))
        return np.where(np.isfinite(acc), acc, 0).astype(np.float32)
    if method == "stdproj":
        total = np.zeros(shape, np.float64)
        total_squared = np.zeros(shape, np.float64)
        for depth in range(d0, d1):
            layer = np.asarray(vol[depth, y0:y1, x0:x1]).astype(np.float64)
            total += layer
            total_squared += layer * layer
        count = max(d1 - d0, 1)
        mean = total / count
        return np.sqrt(np.clip(total_squared / count - mean * mean, 0, None)).astype(np.float32)
    if method == "midslice":
        return np.asarray(vol[(d0 + d1) // 2, y0:y1, x0:x1]).astype(np.float32)
    raise ValueError(f"unknown composite method: {method}")


def _display_map(proj, mask, mode, use_clahe):
    """map a float projection to uint8 for display"""
    if mode == "raw":
        image = np.clip(proj, 0, 255).astype(np.uint8)
    else:
        lo, hi = np.percentile(proj[mask], [1, 99]) if mask.any() else (float(proj.min()), float(proj.max()))
        display = np.clip((proj - lo) / max(hi - lo, 1e-6), 0, 1)
        image = (display * 255).astype(np.uint8)
    if use_clahe:
        image = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image)
    return image * mask.astype(np.uint8)


def make_composite(scroll_id, bounds, method=None, d0=None, d1=None, use_cache=True, display=None, clahe=None):
    """return a uint8 fiber composite restricted to the data bbox"""
    method = method or COMPOSITE_METHOD
    display = display or COMPOSITE_DISPLAY
    clahe = COMPOSITE_CLAHE if clahe is None else clahe
    vol = zarr.open(os.path.join(Config().data.zarr_path, f"{scroll_id}.zarr"), mode="r")
    depth_count = int(vol.shape[0])
    d0 = COMPOSITE_D0 if d0 is None else d0
    d1 = (COMPOSITE_D1 if COMPOSITE_D1 is not None else depth_count) if d1 is None else d1
    d0, d1 = max(0, d0), min(d1, depth_count)
    y0, y1, x0, x1 = bounds
    display_tag = f"_{display}" + ("_clahe" if clahe else "")
    mask_path = mask_path_for(scroll_id)
    if is_extra(scroll_id):
        # extras cache one full-frame composite in composite_cache/extras (R2-synced by assemble_test_segments.py)
        cache_path = os.path.join(COMPOSITE_CACHE, EXTRAS_SUBDIR, f"{scroll_id}_{method}_{d0}-{d1}_full{display_tag}.png")
        if not (use_cache and not GENERATE_COMPOSITES and os.path.exists(cache_path)):
            full_mask = np.array(Image.open(mask_path).convert("L")) > 0
            image = _display_map(_project_depth(vol, d0, d1, method), full_mask, display, clahe)
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            Image.fromarray(image).save(cache_path)
            print(f"[composite] {scroll_id} {method} d{d0}-{d1} full{display_tag} -> {cache_path} {image.shape}")
        return np.array(Image.open(cache_path).convert("L"))[y0:y1, x0:x1]
    bounds_tag = f"y{y0}-{y1}_x{x0}-{x1}"
    cache_path = os.path.join(
        COMPOSITE_CACHE,
        f"{scroll_id}_{method}_{d0}-{d1}_{bounds_tag}{display_tag}.png",
    )
    if use_cache and not GENERATE_COMPOSITES and os.path.exists(cache_path):
        return np.array(Image.open(cache_path).convert("L"))
    full_mask = (np.array(Image.open(mask_path).convert("L")) > 0) if os.path.exists(mask_path) else None
    proj = _project_depth(vol, d0, d1, method, bounds=bounds)
    mask = full_mask[y0:y1, x0:x1] if full_mask is not None else (proj > 0)
    image = _display_map(proj, mask, display, clahe)
    Image.fromarray(image).save(cache_path)
    print(f"[composite] {scroll_id} {method} d{d0}-{d1} {bounds_tag}{display_tag} -> {cache_path} {image.shape}")
    return image


In [ ]:
# inference (the author's ink8um.inference.predict_stack, unmodified) and reusable figure builder
from IPython.display import display
from matplotlib.patches import Rectangle
# plain-text bar: tqdm.auto's ipywidget bar fails in the VS Code notebook renderer
from tqdm import tqdm

# matplotlib/agg and jpeg both cap each image dimension below 2**16 px
MAX_FIGURE_PX = 65000
PREVIEW_DOWNSCALE = 3
FIGURE_DPI = 100
# inverted grayscale: 0 (papyrus) = white, 1 (ink) = black; masked-out cells pale blue
PRED_CMAP = plt.get_cmap("gray_r").copy()
PRED_CMAP.set_bad((0.78, 0.86, 0.95))


def build_config():
    config = Config()
    config.device = "cuda" if torch.cuda.is_available() else "cpu"
    return config


def load_model(config):
    # equivalent to InkDetector.from_pretrained on the local copy: config.json kwargs + strict fp32 weights
    model = InkDetector(**V8_CONFIG)
    model.load_state_dict(load_safetensors(str(MODEL_PATH)), strict=True)
    model = model.to(config.device).eval()
    parameter_count = sum(p.numel() for p in model.parameters())
    print(f"loaded {MODEL_PATH} params={parameter_count:,}")
    return model


def _mask_bbox(mask_bool, margin, align):
    """return an outward tile-aligned bbox around nonzero data"""
    ys, xs = np.where(mask_bool)
    if ys.size == 0:
        return 0, mask_bool.shape[0], 0, mask_bool.shape[1]
    align = max(1, int(align))
    y0 = max(0, ((int(ys.min()) - margin) // align) * align)
    x0 = max(0, ((int(xs.min()) - margin) // align) * align)
    y1 = min(mask_bool.shape[0], ((int(ys.max()) + 1 + margin + align - 1) // align) * align)
    x1 = min(mask_bool.shape[1], ((int(xs.max()) + 1 + margin + align - 1) // align) * align)
    return y0, y1, x0, x1


def load_stack(config, scroll_id):
    """read_stack's full-frame H x W x 24 array: the centred 24 layers, clipped to [0, clip_max], uint8"""
    import time as _time

    source = zarr.open(os.path.join(config.data.zarr_path, f"{scroll_id}.zarr"), mode="r")
    depth_count = int(source.shape[0])
    if depth_count < V8_DEPTH:
        raise RuntimeError(f"{scroll_id} has {depth_count} layers; v8-in needs {V8_DEPTH}")
    layer_start = (depth_count - V8_DEPTH) // 2
    t0 = _time.time()
    print(f"[preload] {scroll_id} layers {layer_start}-{layer_start + V8_DEPTH} of {tuple(source.shape)} -> RAM ... ",
          end="", flush=True)
    window = np.asarray(source[layer_start:layer_start + V8_DEPTH])
    print(f"done in {_time.time() - t0:.1f}s", flush=True)
    # renders sample the 8-bit scan volumes (test fragments are stored as uint16, extras as uint8)
    if int(window.max()) > 255:
        raise RuntimeError(f"{scroll_id} holds values above 255; v8-in expects 8-bit layers")
    return np.ascontiguousarray(np.clip(window, 0, V8_CLIP).astype(np.uint8).transpose(1, 2, 0))


def predict_map(model, config, scroll_id, bounds):
    """predict the whole surface with the author's predict_stack, then crop to the figure bounds"""
    stack = load_stack(config, scroll_id)
    with tqdm(desc=f"test_{scroll_id}", unit="tile") as bar:
        def _progress(done, total):
            bar.total = total
            bar.update(done - bar.n)

        prediction = predict_stack(model, stack, reverse=REVERSE_LAYERS, stride=EVAL_STRIDE, batch_size=BATCH_SIZE,
                                   num_workers=NUM_WORKERS, device=config.device, progress=_progress)
    # display only: pixels no tile reached are drawn as masked (predict_stack leaves them 0)
    covered = np.zeros(prediction.shape, bool)
    for x, y in tile_positions(coverage_mask(stack), V8_TILE, EVAL_STRIDE):
        covered[y:y + V8_TILE, x:x + V8_TILE] = True
    prediction[~covered] = np.nan
    y0, y1, x0, x1 = bounds
    return prediction[y0:y1, x0:x1]


def _run_name():
    return RUN_ID


def _draw_scale_bar(ax, panel_hw, px_scale, font_pt):
    if width >= height:
        x1, y = width - pad, height - pad
        x0 = max(0.0, x1 - length)
        ax.add_patch(Rectangle((x0, y - thickness), x1 - x0, thickness, color="red"))
        ax.text(x0, y - 3 * thickness, "1 cm", color="red", fontsize=font_pt, ha="left", va="bottom")
    else:
        y1, x = height - pad, width - pad
        y0 = max(0.0, y1 - length)
        ax.add_patch(Rectangle((x - thickness, y0), thickness, y1 - y0, color="red"))
        ax.text(x, y0 - 2 * thickness, "1 cm", color="red", fontsize=font_pt, ha="right", va="bottom")


def _save_figure(rows, crop_hw, out_path):
    """lay out (title, image, kind) rows on a pixel-exact grid and save it"""
    crop_height, crop_width = crop_hw
    n_rows, n_cols = len(rows), max(len(row) for row in rows)
    gap = max(PAD_PX, crop_width // 40)
    title_h = max(60, crop_height // 14)
    fig_w = n_cols * crop_width + (n_cols + 1) * gap
    fig_h = n_rows * (title_h + crop_height) + (n_rows + 1) * gap
    scale = min(1.0, MAX_FIGURE_PX / fig_w, MAX_FIGURE_PX / fig_h)
    panel_hw = (max(1, int(crop_height * scale)), max(1, int(crop_width * scale)))
    title_pt = title_h * scale * 0.45 * 72 / FIGURE_DPI
    fig = plt.figure(figsize=(fig_w * scale / FIGURE_DPI, fig_h * scale / FIGURE_DPI), dpi=FIGURE_DPI, facecolor="white")
    for r, row in enumerate(rows):
        row_top = gap + r * (title_h + crop_height + gap) + title_h
        for c, (title, image, kind) in enumerate(row):
            ax = fig.add_axes([
                (gap + c * (crop_width + gap)) / fig_w,
                1 - (row_top + crop_height) / fig_h,
                crop_width / fig_w,
                crop_height / fig_h,
            ])
            if kind.startswith("composite"):
                small = cv2.resize(image, (panel_hw[1], panel_hw[0]), interpolation=cv2.INTER_AREA)
                ax.imshow(small, cmap="gray", vmin=0, vmax=255, interpolation="nearest", aspect="auto")
                if kind == "composite_scale":
                    _draw_scale_bar(ax, panel_hw, panel_hw[1] / crop_width, title_pt)
            else:
                ax.imshow(np.ma.masked_invalid(image), cmap=PRED_CMAP, vmin=0, vmax=1,
                          interpolation="nearest", aspect="auto")
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(title, fontsize=title_pt, pad=title_pt * 0.4)
    is_jpeg = str(out_path).lower().endswith((".jpg", ".jpeg"))
    fig.savefig(out_path, dpi=FIGURE_DPI, facecolor="white", pil_kwargs={"quality": 92} if is_jpeg else None)
    plt.close(fig)


def render_fragment(name, scroll_id, model=None, config=None, fast_path=None, show=True, fast_vertical=False):
    """fast_path overrides <out_dir>/<name>_fast.jpg; show=False skips the inline preview;
    fast_vertical stacks the figure as prediction over composite"""
    mask_full = np.array(Image.open(mask_path_for(scroll_id)).convert("L"))
    bounds = _mask_bbox(mask_full > 0, MASK_CROP_MARGIN, TILE_SIZE)
    y0, y1, x0, x1 = bounds
    crop_height, crop_width = y1 - y0, x1 - x0
    composite = make_composite(scroll_id, bounds=bounds)
    if composite.shape != (crop_height, crop_width):
        composite = cv2.resize(composite, (crop_width, crop_height), interpolation=cv2.INTER_AREA)
    if DRY_RUN or model is None:
        prediction = np.full((crop_height, crop_width), np.nan, np.float32)
        kind = "composite dry run"
    else:
        prediction = predict_map(model, config, scroll_id, bounds)
        kind = "fast"
    out_dir = os.path.join(OUTPUT_DIR, "test_visualizations", _run_name())
    os.makedirs(out_dir, exist_ok=True)
    fast_path = fast_path or os.path.join(out_dir, f"{name}_fast.jpg")
    os.makedirs(os.path.dirname(fast_path), exist_ok=True)
    fast_pred = ("v8-in prediction, centred 24 layers", prediction, "pred")
    composite_scale = ("composite view", composite, "composite_scale")
    fast_rows = [[fast_pred], [composite_scale]] if fast_vertical else [[fast_pred, composite_scale]]
    _save_figure(fast_rows, (crop_height, crop_width), fast_path)
    print(f"[saved] {fast_path} ({kind}) bbox={bounds}")
    if show:
        with Image.open(fast_path) as saved:
            preview = saved.resize(
                (max(1, saved.width // PREVIEW_DOWNSCALE), max(1, saved.height // PREVIEW_DOWNSCALE)),
                Image.LANCZOS,
            )
        display(preview)
    del fast_rows, prediction
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [6]:
# build the v8-in model and load once
import subprocess
from utils.platform import get_zarr_dir

C = build_config()
C.data.zarr_path = get_zarr_dir()
MODEL = None if DRY_RUN else load_model(C)
MASK_DIR = os.path.join(REPO, "masks")
# set by render_extras so helper subprocesses don't print into the notebook
SUBPROCESS_QUIET = False


def _subprocess_output():
    return {"stdout": subprocess.DEVNULL, "stderr": subprocess.DEVNULL} if SUBPROCESS_QUIET else {}


def ensure_assembled(scroll_id):
    """render the test zarr from its tifxyz mesh when it (or its completion mask) is absent"""
    zarr_path = os.path.join(C.data.zarr_path, f"{scroll_id}.zarr")
    # the assembler writes the mask last, so a zarr without a mask is an interrupted render
    if os.path.isdir(zarr_path) and os.path.isfile(os.path.join(REPO, mask_path_for(scroll_id))):
        return
    print(f"[assemble] {scroll_id}: zarr missing or incomplete -> rendering into {C.data.zarr_path}", flush=True)
    subprocess.run(
        [
            sys.executable, os.path.join(REPO, "assemble_test_segments.py"),
            "--only", str(scroll_id),
            "--out-dir", C.data.zarr_path,
        ] + (["--include-extras"] if is_extra(scroll_id) else []),
        cwd=REPO,
        check=True,
        **_subprocess_output(),
    )
    if not os.path.isdir(zarr_path):
        raise FileNotFoundError(f"assembly did not produce {zarr_path}")


def ensure_fragment_ready(scroll_id):
    """assemble and refresh the mask (only used for the crop bounds, as in test_inference.ipynb)"""
    ensure_assembled(scroll_id)
    volume = zarr.open(os.path.join(C.data.zarr_path, f"{scroll_id}.zarr"), mode="r")
    return load_or_create_midslice_mask(
        volume, os.path.join(REPO, mask_path_for(scroll_id)), refresh=True
    )


print(
    "model:", "skipped" if MODEL is None else f"v8-in {V8_VARIANT}",
    "| zarr:", C.data.zarr_path,
    "| window:", f"{V8_DEPTH}x{V8_TILE}x{V8_TILE}",
    "| stride:", EVAL_STRIDE,
    "| batch:", BATCH_SIZE,
    "| reverse:", REVERSE_LAYERS,
)


loaded /vesuvius/models/v8in/base/model.safetensors params=83,317,827
model: v8-in base | zarr: /vesuvius/ves_zarrs2 | window: 24x64x64 | stride: 21 | batch: 16 | reverse: False


In [ ]:
# --- test fragment 1: PHerc0813 (updated patch, 33.31cm², 5081×5701) ---
NAME = "auto_grown_20260814140748"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


[mask] wrote /vesuvius/masks/20260814140748.png from zarr layer 14
[preload] 20260814140748 layers 2-26 of (28, 4641, 5081) -> RAM ... done in 5.1s


In [ ]:
# --- test fragment 2: PHerc0211 merged (5 patches, 7181×6501) ---
NAME = "auto_grown_20260717193517"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# --- test fragment 4: PHerc1447 ---
# 51.27cm^2, 6264×8318 (8.64µm src, upsampled to 9.4µm)
NAME = "auto_grown_20250703034159"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# --- test fragment 3: PHerc1203 ---
# 7.9cm^2, 4035×4455
NAME = "auto_grown_20260720090842"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# test fragment 5: PHerc0826 merged
NAME = "auto_grown_20260723112922652_merged"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# --- test fragment 6: PHerc0846A ---
# 22.88cm^2, 5021×7721 (9.362µm src)
NAME = "auto_grown_20260921094413486"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# --- test fragment 7: PHerc0175A ---
# 11.95cm^2, 4501×5141 (8.64µm src)
NAME = "auto_grown_20260918132724424"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# --- test fragment 8: PHerc0306B ---
# 14.13cm^2, 5121×6181 (8.64µm src)
NAME = "auto_grown_20260922073234974"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# --- test fragment 9: PHerc0800 merged ---
# ~23.36cm^2, 6921×8081 (8.64µm src)
NAME = "auto_grown_20260922161631422_merged"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# --- test fragment 10: PHerc1447 second surface ---
# 10.90cm^2, 4901×4621 (8.64µm src)
NAME = "auto_grown_20260925085345806_abf"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# --- test fragment 11: PHerc0800 expanded auto-winder segment ---
# 5.50cm^2, 2321×4141 (8.64µm src)
NAME = "PHerc0800_z9264_w040_abf"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# --- test fragment 12: PHerc0125 expanded auto-winder segment ---
# 2.50cm^2, 1161×3141 (9.362µm src)
NAME = "PHerc0125_z15344_w040_abf"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# --- test fragment 13: PHerc0211 expanded auto-winder segment ---
# 3.32cm^2, 1941×3641 (9.362µm src)
NAME = "PHerc0211_z7312_w080_abf"
ensure_fragment_ready(FRAGMENTS[NAME])
render_fragment(NAME, FRAGMENTS[NAME], MODEL, C)


In [ ]:
# --- extras setup: fast predictions -> test_visualizations/<run>/extras/<name>.png (no inline preview) ---
# assembles missing zarrs and pulls/generates surface labels + composites for every extra set in one pass
# (R2 env vars needed only for fetching/uploading)
import contextlib
import io

# plain-text bar: tqdm.auto's ipywidget bar fails in the VS Code notebook renderer
from tqdm import tqdm

subprocess.run(
    [sys.executable, os.path.join(REPO, "assemble_test_segments.py"), "--include-extras",
     "--out-dir", C.data.zarr_path],
    cwd=REPO,
    check=True,
)
EXTRAS_OUT_DIR = os.path.join(OUTPUT_DIR, "test_visualizations", _run_name(), EXTRAS_SUBDIR)
EXTRA_IDS = sorted(
    name for name in os.listdir(EXTRAS_MESH_ROOT)
    if os.path.isfile(os.path.join(EXTRAS_MESH_ROOT, name, "x.tif"))
)
# quick_patches_from_D/<scroll>/<segment>
QUICK_D_IDS = {
    scroll: sorted(
        name for name in os.listdir(os.path.join(QUICK_D_MESH_ROOT, scroll))
        if os.path.isfile(os.path.join(QUICK_D_MESH_ROOT, scroll, name, "x.tif"))
    )
    for scroll in sorted(os.listdir(QUICK_D_MESH_ROOT))
}


def render_extras(scroll, extra_ids=None):
    """fast-render every assembled extra segment of one scroll (atlas by name prefix unless extra_ids is given)"""
    global SUBPROCESS_QUIET
    if extra_ids is None:
        extra_ids = [extra_id for extra_id in EXTRA_IDS if extra_id.split("_")[0] == scroll]
    done, skipped, failed = 0, [], []
    SUBPROCESS_QUIET = True
    try:
        for extra_id in tqdm(extra_ids, desc=scroll, unit="patch"):
            if not (os.path.isdir(os.path.join(C.data.zarr_path, f"{extra_id}.zarr")) and os.path.isfile(mask_path_for(extra_id))):
                skipped.append(extra_id)
                continue
            try:
                # per-segment prints and inner tqdm bars are swallowed; only the outer bar is shown
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    ensure_fragment_ready(extra_id)
                    render_fragment(
                        extra_id, extra_id, MODEL, C,
                        fast_path=os.path.join(EXTRAS_OUT_DIR, f"{extra_id}.png"), show=False,
                        fast_vertical=True,
                    )
                done += 1
            except Exception as exc:
                failed.append((extra_id, str(exc)))
    finally:
        SUBPROCESS_QUIET = False
    print(f"[extras] {scroll}: saved {done}/{len(extra_ids)} -> {EXTRAS_OUT_DIR}; "
          f"not assembled (disk/assembly): {len(skipped)}; failed: {len(failed)}")
    for extra_id, message in failed:
        print(f"  FAILED {extra_id}: {message}")


print(f"{len(EXTRA_IDS)} atlas segments:", {s: sum(i.startswith(s + "_") for i in EXTRA_IDS)
                                            for s in sorted({i.split('_')[0] for i in EXTRA_IDS})})
print("quick_patches_from_D segments:", {scroll: len(ids) for scroll, ids in QUICK_D_IDS.items()})


In [ ]:
# --- extras: PHerc0211 (51 segments) ---
render_extras("PHerc0211")


In [ ]:
# --- extras: PHerc0257 (1 segment) ---
render_extras("PHerc0257")


In [ ]:
# --- extras: PHerc0358 (2 segments) ---
render_extras("PHerc0358")


In [ ]:
# --- extras: PHerc0800 (44 segments) ---
render_extras("PHerc0800")


In [ ]:
# --- extras: PHerc0813 (46 segments) ---
render_extras("PHerc0813")


In [ ]:
# --- extras: PHerc0826 (2 segments) ---
render_extras("PHerc0826")


In [ ]:
# --- extras of interest: fast figures -> test_visualizations/<run>/<name>_fast.jpg ---
extra_interest = [
    "PHerc0125_z15344_w040",
    "PHerc0125_z16544_w080",
    "PHerc0125_z16544_w100",
    "PHerc0211_z7312_w080",
    "PHerc0211_z9712_w060",
    "PHerc0800_z9264_w040",
    "PHerc0800_z9264_w080",
]
for extra_id in extra_interest:
    print(f"\n[interest] {extra_id}", flush=True)
    ensure_fragment_ready(extra_id)
    render_fragment(extra_id, extra_id, MODEL, C, fast_vertical=True)


In [ ]:
# --- quick_patches_from_D: PHerc0846A (22 segments) -> test_visualizations/<run>/extras/<name>.png ---
render_extras("PHerc0846A", QUICK_D_IDS["PHerc0846A"])
